# Musanga V4 — NBAA: Annotation Consistency — exploratory audit

**Purpose:** explore *two distinct annotation tasks* before modelling: (1) five experts' `Musanga / non-Musanga / uncertain` classifications on the same candidate crowns, and (2) independently digitized Musanga-positive crown polygons in six verification circles by **four** annotators. Polygon absence is **not** automatically a non-Musanga vote.

**Scientific safeguards:** exact `verify_id` joins; no ground-truth claims or species inference; no models trained; no changes to V3_1/V4 reference or folds; spatial coordinates reprojected to UTM 35N before area/overlap calculations; independent polygon sets are *not* matched by `crown_id` (local per-annotator IDs). Comparisons of polygon overlap are restricted to circles both experts annotated. Shape uncertainty and annotation completeness are flagged, not silently resolved.

Read-only source: `Final/Data/Annotations/Verification/`. All outputs: `Final/Outputs/v4/NBAA_annotation_consistency/`.

**Run cells sequentially in Colab.** Large mosaics and LAS files are not accessed. Sections 00–11 are exploratory; Sections 12–13 fit and evaluate annotator-specific segmentation models on assumed exhaustive mapped circles.

## 00 — V3_1-compatible Drive setup and file inventory

In [ ]:
%pip -q install geopandas shapely pyogrio scipy scikit-learn pandas matplotlib
from pathlib import Path
import json, re, math, warnings, itertools
from collections import Counter, defaultdict
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from shapely.geometry import shape
from shapely.ops import unary_union
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import cohen_kappa_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None,'Original Musanga Final directory unavailable'
DATA=ROOT/'Data'
SRC=DATA/'Annotations'/'Verification'
assert SRC.is_dir(),f'Expected input directory missing: {SRC}'
OUT=ROOT/'Outputs/v4/NBAA_annotation_consistency'
OUT.mkdir(parents=True,exist_ok=True)
files=sorted(p for p in SRC.iterdir() if p.is_file())
inventory=pd.DataFrame([dict(filename=p.name,extension=p.suffix.lower() or '(none)',
                             size_bytes=p.stat().st_size) for p in files])
inventory.to_csv(OUT/'NBAA_00_file_inventory.csv',index=False)
display(inventory)
print('Read-only source:',SRC)
print('Write-only NBAA output:',OUT)
print('Reference V3_1 / V4 datasets are NOT edited.')

## 01 — Classification CSV audit: exact crown IDs, labels and missingness

In [ ]:
CLASS_CSV=sorted(SRC.glob('musanga_classifications_*.csv'))
assert len(CLASS_CSV)==5, f'Expected 5 independent CSVs, found {[p.name for p in CLASS_CSV]}'
def who(p):
    n=p.name.casefold()
    if 'nadine' in n:return 'Nadine'
    if 'marjin' in n or 'marijn' in n:return 'Marijn'
    if 'joseph' in n:return 'Joseph'
    if 'thomas' in n:return 'Thomas'
    if 'kristof' in n:return 'Kristof'
    raise ValueError('Unknown annotator '+str(p))
NORM={'musanga':'M','non_musanga':'N','non-musanga':'N','non musanga':'N','uncertain':'U',
      'm':'M','n':'N','u':'U'}
raw_tables={};qa_rows=[];table_rows=[]
for p in CLASS_CSV:
    person=who(p);d=pd.read_csv(p,dtype={'verify_id':str})
    required={'verify_id','circle_id','classification'}
    assert required.issubset(d.columns),(p.name,required-set(d.columns))
    d['verify_id']=d.verify_id.astype(str).str.strip()
    d['label']=d.classification.astype(str).str.strip().str.casefold().map(NORM)
    qa_rows.append(dict(annotator=person,file=p.name,n_rows=len(d),
        unique_ids=d.verify_id.nunique(),duplicate_ids=int(d.verify_id.duplicated().sum()),
        invalid_labels=int(d.label.isna().sum()),circles=','.join(map(str,sorted(d.circle_id.dropna().unique()))),
        complete_flag_values=str(sorted(d.annotation_complete.dropna().astype(str).unique().tolist()))
             if 'annotation_complete' in d else '(absent)'))
    if d.verify_id.duplicated().any():raise ValueError(f'Duplicate verify_id in {p.name}')
    if d.label.isna().any():raise ValueError(f'Unexpected classification labels in {p.name}')
    raw_tables[person]=d
    table_rows.append(d[['verify_id','circle_id','label']].rename(columns={'circle_id':f'circle_{person}',
                                                                          'label':person}))
QA=pd.DataFrame(qa_rows).sort_values('annotator')
display(QA)
QA.to_csv(OUT/'NBAA_01_classification_file_audit.csv',index=False)
sets={who(p):set(raw_tables[who(p)].verify_id) for p in CLASS_CSV}
all_ids=set.union(*sets.values());common=set.intersection(*sets.values())
print('Union of IDs:',len(all_ids),'IDs present for every annotator:',len(common))
if len(common)!=len(all_ids):
    print('Missing IDs by annotator:',{name:sorted(all_ids-ids) for name,ids in sets.items()})
    print('Only complete matched IDs will enter five-expert agreement summaries.')
all_table=pd.DataFrame({'verify_id':sorted(all_ids)})
for t in table_rows:all_table=all_table.merge(t,on='verify_id',how='left',validate='one_to_one')
for person in raw_tables:
    derived=all_table.verify_id.str.extract(r'^C(\d+)_V\d+$')[0].astype('Int64')
    bad=all_table[f'circle_{person}'].notna()&(derived!=all_table[f'circle_{person}'].astype('Int64'))
    assert not bad.any(),f'circle_id vs verify_id disagreement for {person}'
all_table['circle_id']=all_table.verify_id.str.extract(r'^C(\d+)_V\d+$')[0].astype('Int64')
EXPERTS=['Nadine','Thomas','Joseph','Marijn','Kristof']
assert set(EXPERTS)==set(raw_tables)
matched=all_table.dropna(subset=EXPERTS).copy()
matched[['verify_id','circle_id',*EXPERTS]].to_csv(OUT/'NBAA_01_aligned_labels.csv',index=False)
display(matched.groupby('circle_id').size().rename('candidate_crowns').reset_index())
display(pd.DataFrame({person:matched[person].value_counts() for person in EXPERTS}).fillna(0).astype(int))
print('No label harmonization beyond the explicit M/N/U code mapping.')

## 02 — Classification consistency (descriptive, not annotator 'accuracy')

In [ ]:
stats=[]
for a,b in itertools.combinations(EXPERTS,2):
    q=matched[[a,b]].dropna();definite=q[(q[a]!='U')&(q[b]!='U')]
    stats.append(dict(annotator_a=a,annotator_b=b,n=len(q),
        agreement=float((q[a]==q[b]).mean()),
        cohen_kappa=float(cohen_kappa_score(q[a],q[b],labels=['M','N','U'])),
        n_definite=len(definite),
        definite_agreement=float((definite[a]==definite[b]).mean()) if len(definite) else np.nan,
        definite_kappa=float(cohen_kappa_score(definite[a],definite[b],labels=['M','N']))
           if len(definite) and definite[a].nunique()>1 and definite[b].nunique()>1 else np.nan))
PAIR=pd.DataFrame(stats)
PAIR.to_csv(OUT/'NBAA_02_pairwise_classifications.csv',index=False)
display(PAIR.round(3))
votes=matched[EXPERTS].apply(lambda r:r.value_counts(),axis=1).fillna(0).astype(int)
for k in ['M','N','U']:
    matched[f'votes_{k}']=votes[k].values if k in votes else 0
matched['largest_agreement']=matched[[f'votes_{x}' for x in 'MNU']].max(axis=1)
matched['majority_label']=matched.apply(lambda r: next((k for k in 'MNU' if r[f'votes_{k}']>=3),'none'),axis=1)
display(matched.groupby(['circle_id','largest_agreement']).size().unstack(fill_value=0))
display(matched.groupby('majority_label').size().rename('crowns'))
matched.to_csv(OUT/'NBAA_02_crown_vote_table.csv',index=False)
print('NOTE: agreement measures label similarity, NOT botanical species correctness.')

## 03 — Polygon geometry audit and in-memory topology repair

Some digitized polygons are self-intersecting or topologically invalid. Preserve original geometry quality in a CSV and use `make_valid` on an in-memory analytical copy before unions and IoU. Geometry changes are quantified; source files are never modified.

In [ ]:
POLY_FILES=sorted(p for p in SRC.iterdir() if p.name.startswith('Musanga_Polygones_'))
print('Polygon export files:',[p.name for p in POLY_FILES])
assert len(POLY_FILES)==4, f'Expected 4 polygon exports; found {len(POLY_FILES)}'
from shapely import make_valid
from shapely.geometry import MultiPolygon, Polygon, GeometryCollection
from shapely.ops import unary_union

def polygonal_parts(geom):
    """Preserve only actual polygon components after GEOS repair."""
    if geom is None or geom.is_empty:return []
    if isinstance(geom,Polygon):return [geom]
    if isinstance(geom,MultiPolygon):return list(geom.geoms)
    if hasattr(geom,'geoms'):
        return [p for g in geom.geoms for p in polygonal_parts(g)]
    return []

def repair_polygon_for_analysis(geom):
    """Only an in-memory analysis copy; never overwrite or silently fix source."""
    if geom is None or geom.is_empty:return None
    fixed=make_valid(geom) if not geom.is_valid else geom
    polys=[p for p in polygonal_parts(fixed) if p.is_valid and p.area>0]
    if not polys:return None
    result=unary_union(polys)
    return result if not result.is_empty else None

def safe_union(geometries):
    cleaned=[g for g in geometries if g is not None and not g.is_empty]
    if not cleaned:return GeometryCollection()
    return unary_union(cleaned)

POLY={}
poly_quality=[]
GEOM_QA=[]
UTM='EPSG:32635'
for p in POLY_FILES:
    person=who(p)
    raw=json.loads(p.read_text(encoding='utf-8-sig'))
    assert raw.get('type')=='FeatureCollection' and isinstance(raw.get('features'),list),p.name
    entries=[]
    for i,f in enumerate(raw['features']):
        props=f.get('properties') or {}
        geom=shape(f['geometry']) if f.get('geometry') else None
        entries.append({**props,'geometry':geom,'source_row':i})
    orig=gpd.GeoDataFrame(entries,geometry='geometry',crs='EPSG:4326')
    assert 'circle_id' in orig and 'crown_id' in orig,p.name
    orig['circle_id']=pd.to_numeric(orig.circle_id,errors='raise').astype(int)
    # Work in metric CRS. Keep the raw geometry independently for audit.
    orig=orig.to_crs(UTM)
    clean_geoms=[]
    for _,r in orig.iterrows():
        geom=r.geometry
        was_valid=(geom is not None and not geom.is_empty and geom.is_valid)
        original_area=float(geom.area) if geom is not None and not geom.is_empty else 0.
        fixed=repair_polygon_for_analysis(geom)
        repaired_area=float(fixed.area) if fixed is not None else np.nan
        GEOM_QA.append(dict(annotator=person,source_row=int(r.source_row),
             circle_id=int(r.circle_id),crown_id=str(r.crown_id),
             original_valid=bool(was_valid),original_geom_type=geom.geom_type if geom is not None else '',
             original_area_m2=original_area,repaired_area_m2=repaired_area,
             area_change_pct=100*(repaired_area-original_area)/original_area if original_area>0 else np.nan,
             geometry_usable=bool(fixed is not None),
             repaired=bool(not was_valid and fixed is not None)))
        clean_geoms.append(fixed)
    # All downstream area/IoU/matching/visualization operations use validated copies.
    d=orig.copy()
    d['geometry']=gpd.GeoSeries(clean_geoms,index=d.index,crs=UTM)
    if d.geometry.isna().any():
        print('WARNING: unusable polygon geometry excluded from spatial comparisons:',
              person,d.loc[d.geometry.isna(),['circle_id','crown_id','source_row']].to_dict('records'))
        d=d.loc[d.geometry.notna()].copy()
    assert d.geometry.is_valid.all(),f'Invalid geometry remains for {person}'
    d['area_m2']=d.geometry.area
    d['centroid_x']=d.geometry.centroid.x
    d['centroid_y']=d.geometry.centroid.y
    d['annotator']=person
    POLY[person]=d
    for cid,g in d.groupby('circle_id'):
        geom_union=safe_union(g.geometry.tolist())
        rows_for_circle=[x for x in GEOM_QA if x['annotator']==person and x['circle_id']==int(cid)]
        poly_quality.append(dict(annotator=person,circle_id=int(cid),n_polygons=len(g),
             sum_polygon_area_m2=float(g.area_m2.sum()),
             union_area_m2=float(geom_union.area),
             median_polygon_area_m2=float(g.area_m2.median()),
             duplicate_crown_ids=int(g.crown_id.duplicated().sum()),
             n_invalid_original=sum(not x['original_valid'] for x in rows_for_circle),
             n_repaired=sum(x['repaired'] for x in rows_for_circle),
             n_unusable=sum(not x['geometry_usable'] for x in rows_for_circle),
             annotation_status=str(g.annotation_status.value_counts().to_dict())
                 if 'annotation_status' in g else '(absent)'))
GEOM_AUDIT=pd.DataFrame(GEOM_QA)
GEOM_AUDIT.to_csv(OUT/'NBAA_03_geometry_repair_audit.csv',index=False)
POLY_QA=pd.DataFrame(poly_quality).sort_values(['circle_id','annotator'])
POLY_QA.to_csv(OUT/'NBAA_03_polygon_coverage_by_circle.csv',index=False)
display(POLY_QA.round(2))
print('=== Original geometry validity and in-memory repairs ===')
display(GEOM_AUDIT.groupby('annotator').agg(
    original_invalid=('original_valid',lambda x:int((~x).sum())),
    repaired=('repaired','sum'),unusable=('geometry_usable',lambda x:int((~x).sum())),
    max_abs_area_change_pct=('area_change_pct',lambda x:float(x.abs().max()))).round(3))
matrix=POLY_QA.pivot(index='circle_id',columns='annotator',values='n_polygons').fillna(0).astype(int)
display(matrix)
print('Usable polygon counts:',{a:len(d) for a,d in POLY.items()})
print('All original source files are unchanged. Subsequent sections use repaired analytical copies.')
print('Crucial: missing polygon != non-Musanga classification; completeness must be established.')


## 04 — Pairwise *spatial* agreement where both annotators drew polygons

In [ ]:
# Area-level IoU on comparable circle-specific polygon unions.
# Detection overlap alone cannot distinguish two species without independent truth.
pair_rows=[]
for a,b in itertools.combinations(POLY.keys(),2):
    overlap_circles=sorted(set(POLY[a].circle_id)&set(POLY[b].circle_id))
    for cid in overlap_circles:
        aa=POLY[a].loc[POLY[a].circle_id==cid]
        bb=POLY[b].loc[POLY[b].circle_id==cid]
        ga=unary_union(aa.geometry.tolist());gb=unary_union(bb.geometry.tolist())
        inter=ga.intersection(gb).area;uni=ga.union(gb).area
        pair_rows.append(dict(circle_id=cid,annotator_a=a,annotator_b=b,
          n_a=len(aa),n_b=len(bb),area_a_m2=float(ga.area),area_b_m2=float(gb.area),
          intersection_m2=float(inter),union_m2=float(uni),
          area_iou=float(inter/uni) if uni>0 else np.nan,
          a_covered_by_b=float(inter/ga.area) if ga.area>0 else np.nan,
          b_covered_by_a=float(inter/gb.area) if gb.area>0 else np.nan))
SPATIAL=pd.DataFrame(pair_rows)
SPATIAL.to_csv(OUT/'NBAA_04_pairwise_polygon_union_IoU.csv',index=False)
display(SPATIAL.sort_values(['circle_id','annotator_a','annotator_b']).round(3))
display(SPATIAL.groupby(['annotator_a','annotator_b']).agg(
  circles=('circle_id','count'),mean_area_IoU=('area_iou','mean'),
  median_area_IoU=('area_iou','median')).round(3))
print('These are unsigned overlap measures; polygons can represent different crowns or delineation rules.')

## 05 — Crown-to-crown spatial matching (exploratory, threshold sensitivity)

In [ ]:
def match_iou(A,B,minimum):
    # IoU is computed in projected metres; optimal one-to-one Hungarian assignment.
    if not len(A) or not len(B):return dict(matches=0,iou_mean=np.nan,frac_a=0,frac_b=0)
    mat=np.zeros((len(A),len(B)),dtype=float)
    for i,ga in enumerate(A.geometry):
        for j,gb in enumerate(B.geometry):
            if not ga.intersects(gb):continue
            u=ga.union(gb).area
            mat[i,j]=ga.intersection(gb).area/u if u>0 else 0
    rr,cc=linear_sum_assignment(-mat)
    good=mat[rr,cc]>=minimum
    return dict(matches=int(good.sum()),
        iou_mean=float(mat[rr[good],cc[good]].mean()) if good.any() else np.nan,
        frac_a=float(good.sum()/len(A)),frac_b=float(good.sum()/len(B)))
match_rows=[]
for a,b in itertools.combinations(POLY.keys(),2):
    for cid in sorted(set(POLY[a].circle_id)&set(POLY[b].circle_id)):
        da=POLY[a].loc[POLY[a].circle_id==cid]
        db=POLY[b].loc[POLY[b].circle_id==cid]
        for threshold in (.10,.30,.50):
            match_rows.append(dict(circle_id=cid,a=a,b=b,iou_threshold=threshold,
                n_a=len(da),n_b=len(db),**match_iou(da,db,threshold)))
MATCH=pd.DataFrame(match_rows)
MATCH.to_csv(OUT/'NBAA_05_one_to_one_crown_matches.csv',index=False)
display(MATCH.groupby(['a','b','iou_threshold']).agg(
    circles=('circle_id','count'),matched_crowns=('matches','sum'),
    avg_fraction_a=('frac_a','mean'),avg_fraction_b=('frac_b','mean')).round(3))
print('Unmatched crown need not be a false positive: reference completeness is unverified.')

## 06 — Compact site map / visual QA and exploration hand-off

In [ ]:
# Small vector-only visualization: no large TIFF reads.
from matplotlib.patches import Patch
colors={'Nadine':'#1e7962','Thomas':'#cf7c37','Joseph':'#6178b8','Marijn':'#ad589c'}
circles=sorted(set.union(*(set(d.circle_id) for d in POLY.values())))
fig,axes=plt.subplots(2,3,figsize=(15,10))
for ax,cid in zip(axes.flat,circles):
    for person in ['Nadine','Thomas','Joseph','Marijn']:
        d=POLY.get(person)
        if d is None:continue
        subset=d[d.circle_id==cid]
        if len(subset):subset.boundary.plot(ax=ax,color=colors[person],linewidth=1,alpha=.8)
    ax.set_title(f'Circle {cid} — independently digitized crowns')
    ax.set_aspect('equal');ax.tick_params(labelsize=7)
for ax in axes.flat[len(circles):]:ax.axis('off')
fig.legend(handles=[Patch(facecolor='none',edgecolor=colors[n],label=n) for n in colors],
           loc='lower center',ncol=4)
plt.tight_layout(rect=[0,.04,1,1])
fig.savefig(OUT/'NBAA_06_polygon_overlay_by_circle.png',dpi=170)
plt.show();plt.close(fig)
contract=dict(schema='NBAA_v4_exploratory_1',
  source_directory=str(SRC),output_directory=str(OUT),
  classification_files={p:raw_tables[p].shape[0] for p in EXPERTS},
  common_classification_crowns=int(len(matched)),
  digitized_polygon_counts={p:int(len(POLY[p])) for p in POLY},
  classification_policy='Uncertain is a separate observed label; missing is not negative',
  polygon_policy='Independent positive polygons; absent polygons not negative; no automatic truth',
  alignment='CSV verify_id joins; polygon overlays by projected geometry and circle',
  train_test_policy='No training or new splits in this audit',
  further_review=['Digitizing completeness for each circle and annotator',
                  'Compare classification vs polygon tasks only where identity/coverage established',
                  'Independent RGB/DSM embeddings and spatially held-out annotator predictors',
                  'Repeat blinded annotations to assess genuine intra-annotator repeatability'])
(OUT/'NBAA_manifest.json').write_text(json.dumps(contract,indent=2))
print('NBAA exploration complete. Outputs:')
for p in sorted(OUT.glob('NBAA_*')):print(' ',p.name)
print('NEXT DECISION: review counts, completeness, overlap and image QA before modelling.')

## 07 — Which annotators outlined the **same crown**? (exploratory spatial object agreement)

**Critical distinction:** a zero in the polygon count table means **no polygon record**, *not* an observed zero-Musanga survey. Every circle without polygons is marked `not_recorded`; circles with polygons are marked `polygons_present_completion_unverified`, because all source polygon records have `annotation_status='draft'`. Only independently confirmed complete surveys can support false-negative/detection-rate interpretation.

This block identifies spatially overlapping polygon groups. An edge requires **different annotators**, geometrical intersection, polygon IoU ≥0.10 *or* at least 35% of the smaller polygon covered, plus nearby centroids. Connected groups with **multiple polygons from the same annotator** are flagged `ambiguous_group` and *not included in reliable 1/2/3/4 agreement counts*. Results are sensitive to these explicitly shown thresholds. Spatial overlap is not a species-truth vote.

Outputs: per-circle coverage, per-crown member table, group-level number of annotators (1–4), counts by circle, diagnostics/ambiguity, and plots. Visual overlay displays are vector-based; a georeferenced UAV RGB clip is added only if found locally and raster windows can be read safely.

In [ ]:
# NBAA Section 07A — coverage matrix and conservative spatial crown-group graph.
# Only reuses already-loaded POLY (repaired, independent) and reads no mosaics.
from collections import defaultdict
from itertools import combinations
from scipy.spatial.distance import euclidean
from shapely.geometry import box
import networkx as nx
try:
    import rasterio
    from rasterio.windows import from_bounds as raster_window_from_bounds
except ImportError:
    rasterio=None

annotators=['Nadine','Thomas','Joseph','Marijn']
circles=sorted(set().union(*[set(df.circle_id.unique()) for df in POLY.values()]))
coverage=[]
for circle in circles:
    for person in annotators:
        df=POLY[person];part=df[df.circle_id==circle]
        coverage.append(dict(circle_id=int(circle),annotator=person,
          n_polygons=int(len(part)),
          status='polygons_present_completion_unverified' if len(part) else 'not_recorded',
          complete_verified=False))
COVERAGE=pd.DataFrame(coverage)
COVERAGE.to_csv(OUT/'NBAA_07_circle_annotator_coverage.csv',index=False)
display(COVERAGE.pivot(index='circle_id',columns='annotator',values='status'))
display(COVERAGE.pivot(index='circle_id',columns='annotator',values='n_polygons').fillna(0).astype(int))
print('CAUTION: zero records are unassessed, not confirmed zero Musanga.')
print('Even where polygons exist, full-circle completeness is not independently established.')

MIN_IOU=.10
MIN_SMALLER_OVERLAP=.35
MAX_CENTROID_FACTOR=1.5 # times the sum of equivalent-area circle radii
members=[];groups=[];edges=[]
for cid in circles:
    records=[]
    for who_ in annotators:
        d=POLY[who_]
        for _,r in d[d.circle_id==cid].iterrows():
            records.append(dict(annotator=who_,crown_id=str(r.crown_id),
                source_row=int(r.source_row),geometry=r.geometry,
                area_m2=float(r.geometry.area)))
    graph=nx.Graph();graph.add_nodes_from(range(len(records)))
    for i,j in combinations(range(len(records)),2):
        a,b=records[i],records[j]
        if a['annotator']==b['annotator']:continue
        ga,gb=a['geometry'],b['geometry']
        if not ga.intersects(gb):continue
        inter=ga.intersection(gb).area
        if inter<=0:continue
        union=ga.area+gb.area-inter
        iou=inter/union if union else 0
        smaller=inter/min(ga.area,gb.area) if min(ga.area,gb.area)>0 else 0
        ra=np.sqrt(ga.area/np.pi);rb=np.sqrt(gb.area/np.pi)
        dist=ga.centroid.distance(gb.centroid)
        if (iou>=MIN_IOU or smaller>=MIN_SMALLER_OVERLAP) and dist<=MAX_CENTROID_FACTOR*(ra+rb):
            graph.add_edge(i,j,weight=float(iou))
            edges.append(dict(circle_id=int(cid),annotator_a=a['annotator'],annotator_b=b['annotator'],
              crown_a=a['crown_id'],crown_b=b['crown_id'],
              iou=float(iou),smaller_overlap=float(smaller),centroid_distance_m=float(dist)))
    for group_idx,comp in enumerate(sorted(nx.connected_components(graph),key=lambda s:min(s)),start=1):
        rows=[records[i] for i in sorted(comp)]
        unique_people={r['annotator'] for r in rows}
        collisions=[person for person in unique_people if sum(r['annotator']==person for r in rows)>1]
        ambiguous=bool(collisions)
        gid=f'C{cid:02d}_G{group_idx:03d}'
        geoms=[r['geometry'] for r in rows]
        union=safe_union(geoms)
        groups.append(dict(group_id=gid,circle_id=int(cid),
            n_polygons=len(rows),n_annotators=len(unique_people),
            annotators=';'.join(sorted(unique_people)),
            ambiguous_group=ambiguous,
            same_annotator_collisions=';'.join(sorted(collisions)),
            member_ids=';'.join(f"{r['annotator']}:{r['crown_id']}" for r in rows),
            centroid_easting=float(union.centroid.x),
            centroid_northing=float(union.centroid.y),
            union_area_m2=float(union.area)))
        for r in rows:
            members.append(dict(group_id=gid,circle_id=int(cid),
                annotator=r['annotator'],crown_id=r['crown_id'],
                source_row=r['source_row'],polygon_area_m2=r['area_m2'],
                ambiguous_group=ambiguous))
GROUPS=pd.DataFrame(groups)
MEMBERS=pd.DataFrame(members)
EDGES=pd.DataFrame(edges)
for name,frame in [('NBAA_07_crown_groups.csv',GROUPS),
                    ('NBAA_07_crown_members.csv',MEMBERS),
                    ('NBAA_07_spatial_match_edges.csv',EDGES)]:
    frame.to_csv(OUT/name,index=False)
print('Spatial crown groups (not verified true crowns):',len(GROUPS))
print('Ambiguous chained groups:',int(GROUPS.ambiguous_group.sum()))
display(GROUPS.groupby(['n_annotators','ambiguous_group']).size().rename('groups').reset_index())
reliable=GROUPS.loc[~GROUPS.ambiguous_group]
print('Crown groups by number of annotators, excluding ambiguous links:')
display(reliable.groupby(['circle_id','n_annotators']).size().unstack(fill_value=0).reindex(columns=[1,2,3,4],fill_value=0))
print('Full group identity table (inspect before interpreting consensus):')
display(GROUPS[['group_id','circle_id','n_polygons','n_annotators','annotators',
                'ambiguous_group','member_ids']].head(35))


## 08 — High-resolution RGB inspection on original V3_1 UAV mosaics

Uses the **authoritative original RGB paths** from the frozen V3_1 NB00 audit manifest, not pre-cut verification clips. For each circle, it selects the mosaic by georeferenced intersection, reads a window at **native UAV pixel resolution**, overlays the four annotators' crown boundaries, and records the actual source resolution and any required downsampling. No whole mosaics are loaded. The large PNG figures are written to `08_high_resolution_rgb/`.

**Note:** the overlays remain visual QA, not proven correct crown identity. Source TIFFs stay read-only.

In [ ]:
# NBAA Section 08 — original HIGH-RES UAV RGB mosaics from frozen V3_1 NB00 manifest.
# No pre-cut clips are required. Only small exact native-resolution windows are read.
import json, math
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import from_bounds as window_from_bounds, Window, transform as window_transform
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds
from shapely.geometry import box
from matplotlib.patches import Patch
import matplotlib.patheffects as pe

# Authoritative audited RGB source manifest shared by V3_1 and V4 (do not infer paths).
manifest=ROOT/'Outputs/V3_1/00_input_audit/00_input_manifest.json'
assert manifest.is_file(),('V3_1 NB00 manifest missing:',str(manifest))
audit=json.loads(manifest.read_text())
assert audit.get('schema_version')=='v3_1_input_audit_kvo_final',audit.get('schema_version')
RGB_SOURCES={r['file']:Path(r['path']) for r in audit['mosaics']
             if r['file']!='Mos_3_4_transparent_mosaic_group1_georef.tif'}
assert len(RGB_SOURCES)==5 and all(p.is_file() for p in RGB_SOURCES.values()),RGB_SOURCES
print('Using original five audited RGB UAV mosaics:')
for k,p in RGB_SOURCES.items(): print(' ',k,'→',p)

COLORS={'Nadine':'#00ff56','Thomas':'#ffff00','Joseph':'#00eaff','Marijn':'#ff00f5'}
SHORT={'Nadine':'Nad','Thomas':'Thom','Joseph':'Jo','Marijn':'Ma'}
CIRCLE_BOUNDARY_FILE=Path(audit['exhaustive_annotations']['complete_roi_boundaries'])
assert CIRCLE_BOUNDARY_FILE.is_file(),CIRCLE_BOUNDARY_FILE
SURVEY_CIRCLES=gpd.read_file(CIRCLE_BOUNDARY_FILE).reset_index(drop=True)
# V3_1 original contract: circle_id is the ORIGINAL zero-based shapefile row,
# not a pre-existing attribute (see V3_1 NB01 section 04).
assert len(SURVEY_CIRCLES)>=34,len(SURVEY_CIRCLES)
SURVEY_CIRCLES['circle_id']=np.arange(len(SURVEY_CIRCLES),dtype=int)
RGB_OUTPUT=OUT/'08_high_resolution_rgb'
RGB_OUTPUT.mkdir(parents=True,exist_ok=True)
# Native resolution is always retained unless an especially large output
# requires an explicit cap (logged in QA), not silently downsampled.
MAX_OVERVIEW_PX=8500
PAD_M=8.0
overview_rows=[]

def rgb_uint8(x, ds):
    """Convert 3 RGB bands into display RGB; preserve uint8, rescale other types for viewing."""
    if x.dtype==np.uint8:return np.moveaxis(x,0,-1)
    y=x.astype('float32')
    # 16-bit images may use 0..65535; use metadata-independent robust display stretch.
    out=np.empty_like(y,dtype=np.uint8)
    for k in range(min(3,y.shape[0])):
        v=y[k];valid=np.isfinite(v)
        lo,hi=np.nanpercentile(v[valid],[1,99]) if valid.any() else (0,1)
        out[k]=np.uint8(np.clip((v-lo)/(hi-lo if hi>lo else 1)*255,0,255))
    return np.moveaxis(out,0,-1)

def native_window(ds, bounds, pad_m=0):
    l,b,r,t=bounds
    w=window_from_bounds(l-pad_m,b-pad_m,r+pad_m,t+pad_m,transform=ds.transform)
    # Clip to actual image: a circle at a tile edge is never read out-of-bounds.
    w=w.round_offsets().round_lengths()
    return w.intersection(Window(0,0,ds.width,ds.height))

for cid in circles:
    # Each group is already a repaired polygon in UTM35N; find the SOURCE RGB
    # mosaic covering most of the group's extent (works for all four forest sites).
    parts={a:POLY[a].loc[POLY[a].circle_id==cid] for a in COLORS}
    all_geoms=[g for p in parts.values() for g in p.geometry]
    if not all_geoms:continue
    target=safe_union(all_geoms)
    candidates=[]
    for name,path in RGB_SOURCES.items():
        with rasterio.open(path) as ds:
            if ds.crs is None:continue
            xmin,ymin,xmax,ymax=transform_bounds(ds.crs,'EPSG:32635',*ds.bounds)
            frac=target.intersection(box(xmin,ymin,xmax,ymax)).area/target.area
            candidates.append((frac,name,path))
    frac,name,path=max(candidates,key=lambda x:x[0])
    if frac<.90: print(f'WARNING: circle {cid}: best RGB footprint intersection {frac:.1%}; inspect alignment.')
    with rasterio.open(path) as ds:
        if ds.crs is None or ds.count<3:raise ValueError(f'Unexpected RGB source {path}')
        projected={a:p.to_crs(ds.crs) for a,p in parts.items()}
        geometry=gpd.GeoSeries([target],crs='EPSG:32635').to_crs(ds.crs).iloc[0]
        survey=SURVEY_CIRCLES.loc[SURVEY_CIRCLES.circle_id==cid].to_crs(ds.crs)
        assert len(survey)==1, f'Circle boundary {cid} missing'
        circle_geom=survey.geometry.iloc[0]
        win=native_window(ds,circle_geom.bounds,pad_m=PAD_M)
        native_w,native_h=int(win.width),int(win.height)
        factor=max(1,math.ceil(max(native_w,native_h)/MAX_OVERVIEW_PX))
        out_h=max(1,math.ceil(native_h/factor));out_w=max(1,math.ceil(native_w/factor))
        raw=ds.read([1,2,3],window=win,out_shape=(3,out_h,out_w),
                    resampling=Resampling.average if factor>1 else Resampling.nearest)
        arr=rgb_uint8(raw,ds)
        actual_bounds=rasterio.windows.bounds(win,ds.transform)
        left,bottom,right,top=actual_bounds
        fig,ax=plt.subplots(figsize=(14,14*arr.shape[0]/arr.shape[1]),dpi=130)
        ax.imshow(arr,extent=(left,right,bottom,top),interpolation='nearest')
        for a,pg in projected.items():
            if len(pg):
                pg.plot(ax=ax,facecolor=COLORS[a],edgecolor='none',alpha=.09)
                pg.boundary.plot(ax=ax,edgecolor='black',linewidth=3.6,alpha=.95)
                pg.boundary.plot(ax=ax,edgecolor=COLORS[a],linewidth=2.3,alpha=1)
                for pg_geom in pg.geometry:
                    pt=pg_geom.representative_point()
                    ax.text(pt.x,pt.y,SHORT[a],color=COLORS[a],fontsize=9,
                            fontweight='bold',ha='center',va='center',zorder=15,
                            path_effects=[pe.Stroke(linewidth=2.5,foreground='black'),pe.Normal()])
        gpd.GeoSeries([circle_geom],crs=ds.crs).boundary.plot(ax=ax,color='black',linewidth=5,linestyle='--',zorder=18)
        gpd.GeoSeries([circle_geom],crs=ds.crs).boundary.plot(ax=ax,color='white',linewidth=3,linestyle='--',zorder=19)
        for _,r in GROUPS[GROUPS.circle_id==cid].iterrows():
            ctr=gpd.GeoSeries([gpd.points_from_xy([r.centroid_easting],[r.centroid_northing])[0]],
                    crs='EPSG:32635').to_crs(ds.crs).iloc[0]
            ax.text(ctr.x,ctr.y,r.group_id.split('_')[-1]+('!' if r.ambiguous_group else ''),
                color='white',fontsize=7,ha='center',va='center',
                bbox=dict(facecolor='black',alpha=.60,pad=1.0,edgecolor='none'))
        ax.set(xlim=(left,right),ylim=(bottom,top),
              title=f'Circle {cid} | original RGB: {name} | source {native_w}×{native_h}px | read factor {factor}')
        ax.set_aspect('equal')
        fig.legend(handles=[Patch(facecolor=COLORS[a],edgecolor='black',label=f'{SHORT[a]} = {a}') for a in COLORS],
            loc='lower center',ncol=4)
        fig.tight_layout(rect=[0,.025,1,1])
        dest=RGB_OUTPUT/f'circle_{cid:02d}_RGB_polygon_overlay.png'
        fig.savefig(dest,dpi=200)
        plt.show();plt.close(fig)
        overview_rows.append(dict(circle_id=int(cid),source=name,
              source_path=str(path),source_resolution_m=float(abs(ds.transform.a)),
              rgb_coverage_fraction=float(frac),native_window_width_px=native_w,
              native_window_height_px=native_h,read_downsample_factor=factor,
              rendered_width_px=out_w,rendered_height_px=out_h,figure=str(dest)))
OVERVIEW_QA=pd.DataFrame(overview_rows)
OVERVIEW_QA.to_csv(OUT/'NBAA_08_rgb_source_and_resolution_audit.csv',index=False)
display(OVERVIEW_QA[['circle_id','source','source_resolution_m','rgb_coverage_fraction',
                     'native_window_width_px','native_window_height_px','read_downsample_factor']])
print('Full-resolution RGB polygon overlays:',RGB_OUTPUT)
print('Each overlay uses the source RGB raster window; no full mosaic is loaded.')
print('For crown-edge evaluation, use native-resolution source windows rather than stretching small clips.')


## 09 — Cross-task consistency: 55 labelled crowns versus independent digitization

**Source fix:** the original `Verification.shp` does not contain the canonical `verify_id` schema assumed in the first version. This section now reads the **verified GEE canonical** `verification_unique.geojson` (from `Final/Outputs/V3/06_GEE_Verification_v2`) with its explicit `verify_id` and `circle_id` fields. If that file is unavailable, it uses `verification_68_source_mapping.csv` to reconcile raw shapefile rows to canonical crown IDs; it will stop rather than guess. The source has 60 unique candidate polygons; this analysis uses exactly the **55 classified by all five experts**.

Compare class votes against independent positive polygon digitization only where a matched crown is found. Missing survey or no polygon does **not** imply a negative identification. Exact matching is audited, and V3_1/V4 training data are not modified.

In [ ]:
# NBAA Section 09 — conservative candidate polygon versus manually drawn crown.
# The working GEE verification data are keyed by verify_id; the original
# Verification.shp is a raw digitization file and does NOT guarantee this schema.
# Prefer the validated canonical 60-feature GeoJSON produced by GEE Verification v2.
VERIFY_GEOJSON_OVERRIDE=None # optional exact path, if data were moved
VERIFY_DIR_CANDIDATES=[
    ROOT/'Outputs/V3/06_GEE_Verification_v2',   # ACTUAL connected Drive folder
    ROOT/'Outputs/V3_1/06_GEE_Verification_v2',
    ROOT/'Outputs/v3/06_GEE_Verification_v2',
    ROOT/'Outputs/06_GEE_Verification_v2',
    SRC]
paths=([Path(VERIFY_GEOJSON_OVERRIDE)] if VERIFY_GEOJSON_OVERRIDE else [])+[
    folder/'verification_unique.geojson' for folder in VERIFY_DIR_CANDIDATES]
verified_geojson=next((p for p in paths if p.is_file()),None)
if verified_geojson:
    print('Using verified GEE canonical geometry:',verified_geojson)
    src_verify=gpd.read_file(verified_geojson)
else:
    # Fall back to the original raw shapefile, reconciled using the authoritative
    # GEE source-row -> canonical verify_id mapping from the same V3 output folder.
    mapping_paths=[folder/'verification_68_source_mapping.csv' for folder in VERIFY_DIR_CANDIDATES]
    mapping_file=next((p for p in mapping_paths if p.is_file()),None)
    assert mapping_file is not None,(
        'Cannot identify crown geometry by verify_id. Original Verification.shp '
        'does not provide canonical ids. Expected verified_geojson or GEE '
        'verification_68_source_mapping.csv in Outputs/V3/06_GEE_Verification_v2. '
        'Set VERIFY_GEOJSON_OVERRIDE to the exact verified GeoJSON path.')
    VERIFY=SRC/'Verification.shp'
    assert VERIFY.is_file(),VERIFY
    source=gpd.read_file(VERIFY).reset_index(drop=True)
    source['source_row']=np.arange(len(source))
    mapping=pd.read_csv(mapping_file)
    assert {'source_row','canonical_id','is_duplicate'}.issubset(mapping.columns)
    assert mapping.source_row.is_unique
    assert len(mapping)==len(source),(len(mapping),len(source))
    canonical=mapping.loc[~mapping.is_duplicate.astype(str).str.lower().eq('true'),
                          ['source_row','canonical_id','circle_id']].copy()
    src_verify=source.merge(canonical,on='source_row',how='inner',validate='one_to_one')
    src_verify=src_verify.rename(columns={'canonical_id':'verify_id'})
    print('Using raw shapefile reconciled to GEE canonical ids via:',mapping_file)
print('Geometry source columns:',src_verify.columns.tolist())
assert {'verify_id','circle_id'}.issubset(src_verify.columns),src_verify.columns.tolist()
src_verify['verify_id']=src_verify.verify_id.astype(str).str.strip()
print('Reference source raw:',len(src_verify),'rows /',src_verify.verify_id.nunique(),'unique IDs')
sub=src_verify.loc[src_verify.verify_id.isin(matched.verify_id)].to_crs(UTM).copy()
assert set(sub.verify_id)==set(matched.verify_id),{
  'missing':sorted(set(matched.verify_id)-set(sub.verify_id)),
  'extra':sorted(set(sub.verify_id)-set(matched.verify_id))}
if sub.verify_id.duplicated().any():
    print('Duplicate reference IDs: verify geometry equivalence before deduplication')
    for vid,part in sub.groupby('verify_id'):
        if len(part)>1:
            gg=[repair_polygon_for_analysis(g) for g in part.geometry]
            if any(g is None or not gg[0].equals_exact(g,1e-3) for g in gg[1:]):
                raise ValueError('Conflicting duplicate verify_id geometry: '+vid)
    sub=sub.drop_duplicates('verify_id').copy()
sub['geometry']=[repair_polygon_for_analysis(g) for g in sub.geometry]
assert sub.geometry.notna().all() and len(sub)==55
# Discard any source-side duplicate circle_id column and retain original canonical
# verification circle id. The classification table has the same IDs.
sub['circle_id']=pd.to_numeric(sub.circle_id,errors='raise').astype(int)
REF=sub.merge(matched[['verify_id',*EXPERTS]],on='verify_id',validate='one_to_one')
assert (REF.circle_id==REF.verify_id.str.extract(r'^C(\d+)_V')[0].astype(int)).all()
print('Verified 55 unique reference crown polygons with exact expert-label joins.')
cross=[]
for _,r in REF.iterrows():
    for person in ['Nadine','Thomas','Joseph','Marijn']:
        candidates=POLY[person].loc[POLY[person].circle_id==int(r.circle_id)]
        best=(0.,0.,None)
        for _,q in candidates.iterrows():
            inter=r.geometry.intersection(q.geometry).area
            if inter<=0:continue
            uni=r.geometry.union(q.geometry).area
            iou=inter/uni if uni else 0.
            frac=inter/min(r.geometry.area,q.geometry.area)
            if iou>best[0]:best=(iou,frac,str(q.crown_id))
        match=(best[0]>=.10 or best[1]>=.35)
        status='not_recorded' if len(candidates)==0 else 'polygons_present_completion_unverified'
        label=r[person]
        interpretation=('not_assessed' if status=='not_recorded' else
           'possible_cross_task_conflict' if match and label=='N' else
           'affirmative_concordance' if match and label=='M' else
           'uncertain_and_delineated' if match and label=='U' else
           'unmatched_completeness_unknown')
        cross.append(dict(verify_id=r.verify_id,circle_id=int(r.circle_id),annotator=person,
            classification=label,survey_status=status,matched=bool(match),
            best_polygon_id=best[2],best_iou=float(best[0]),
            fraction_of_smaller_polygon=float(best[1]),interpretation=interpretation))
CROSS=pd.DataFrame(cross)
CROSS.to_csv(OUT/'NBAA_09_cross_task_crown_match.csv',index=False)
display(CROSS.groupby(['annotator','classification','interpretation']).size().rename('n').reset_index())
display(CROSS.groupby(['annotator','classification']).agg(
    n=('verify_id','count'),matched=('matched','sum'),
    median_iou=('best_iou','median')).round(3))
print('Only candidate conflicts are flagged. Absence of polygon is NOT a negative label.')

## 10 — Independent crown features: RGB and DSM (55 common reference objects)

All 55 candidate crown polygons are shared across experts; extract **one** feature vector per reference crown from the native-resolution original RGB, regardless of who selected it. Add area, compactness, RGB chromaticity, within-crown variation, and **SfM DSM canopy-surface heterogeneity** where a validated DSM mosaic exists.

**Controls:** local small windows only, avoid loading the mosaics; no labels in the feature matrix; do not use MS until its radiometry is resolved. Source RGB band intensities are not consistently calibrated across acquisitions. DSM measures are canopy-surface, not terrain-normalized tree height. This is exploratory, not species ground truth.

In [ ]:
# NBAA Section 10: crown features, windowed RGB + optional DSM on RGB grid.
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds as win_from_bounds,Window
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from shapely.geometry import mapping
DSM_REG=ROOT/'Outputs/v4/01_reference_data_no_MOS34/01_dsm_file_registry.json'
DSM_PATHS={}
if DSM_REG.is_file():
    DSM_PATHS={k:Path(v) for k,v in json.loads(DSM_REG.read_text()).get('rgb_to_dsm',{}).items()}
RGB_BY_CIRCLE=OVERVIEW_QA.set_index('circle_id')['source'].to_dict()
assert set(REF.circle_id.astype(int)).issubset(RGB_BY_CIRCLE)
feats=[]
for _,r in REF.iterrows():
    rgbname=RGB_BY_CIRCLE[int(r.circle_id)]
    with rasterio.open(RGB_SOURCES[rgbname]) as rgb:
        geom=gpd.GeoSeries([r.geometry],crs=UTM).to_crs(rgb.crs).iloc[0]
        w=win_from_bounds(*geom.bounds,transform=rgb.transform).round_offsets().round_lengths()
        w=w.intersection(Window(0,0,rgb.width,rgb.height))
        if w.width<2 or w.height<2:continue
        tr=rasterio.windows.transform(w,rgb.transform)
        a=rgb.read([1,2,3],window=w,masked=True).astype('float64')
        mask=geometry_mask([mapping(geom)],out_shape=(int(w.height),int(w.width)),
                           transform=tr,invert=True)
        mask &= (~np.ma.getmaskarray(a)).all(axis=0)
        if mask.sum()<20:continue
        v=a.data[:,mask]
        raw=np.median(v,axis=1)
        chrom=v/np.maximum(v.sum(axis=0,keepdims=True),1.)
        means=v.mean(axis=0)
        feature=dict(verify_id=str(r.verify_id),circle_id=int(r.circle_id),mosaic=rgbname,
            area_m2=float(geom.area),compactness=float(4*np.pi*geom.area/max(geom.length**2,1e-10)),
            n_rgb_pixels=int(mask.sum()),RGB_R=float(raw[0]),RGB_G=float(raw[1]),RGB_B=float(raw[2]),
            color_R=float(np.median(chrom[0])),color_G=float(np.median(chrom[1])),
            color_B=float(np.median(chrom[2])),
            brightness_CV=float(np.std(means)/max(np.mean(means),1.)),
            RGB_texture=float(np.std(means)),
            DSM_valid_fraction=np.nan,DSM_roughness_m=np.nan,DSM_range_p90_p10_m=np.nan)
        dp=DSM_PATHS.get(rgbname)
        if dp is not None and dp.exists():
            with rasterio.open(dp) as ds, WarpedVRT(ds,crs=rgb.crs,transform=rgb.transform,
                 width=rgb.width,height=rgb.height,resampling=Resampling.bilinear) as zvrt:
                z=zvrt.read(1,window=w,masked=True)
            zdata=np.asarray(z.data,dtype='float64')
            zvalid=(~np.ma.getmaskarray(z))&np.isfinite(zdata)&(~np.isclose(zdata,0))
            good=zvalid&mask
            feature['DSM_valid_fraction']=float(good.sum()/mask.sum())
            if good.sum()>=20:
                zv=zdata[good]
                feature['DSM_roughness_m']=float(np.std(zv))
                feature['DSM_range_p90_p10_m']=float(np.percentile(zv,90)-np.percentile(zv,10))
        feats.append(feature)
FEATURES=pd.DataFrame(feats)
assert len(FEATURES)>0
FEATURES.to_csv(OUT/'NBAA_10_reference_crown_features.csv',index=False)
missing=sorted(set(matched.verify_id)-set(FEATURES.verify_id))
print(f'Feature extraction: {len(FEATURES)} / {len(matched)} reference crowns.')
if missing: print('WARNING: missing RGB window features:',missing)
display(FEATURES.groupby('circle_id').agg(
    crowns=('verify_id','count'),median_area_m2=('area_m2','median'),
    median_rgb_pixels=('n_rgb_pixels','median'),
    n_dsm=('DSM_roughness_m',lambda x:int(x.notna().sum()))).round(2))
display(FEATURES.head().round(3))

## 11 — Unsupervised visual phenotype clustering and annotator enrichment

Cluster the **55 shared reference crowns**, using only RGB/shape/available DSM features—not human labels or which annotator drew a polygon. Then compare the *same clusters* against all five annotators' positive/negative/uncertain votes. Check whether clusters merely reflect the site/circle, RGB illumination, or crown size. Two to four cluster solutions are explored.

**Scientific caution:** These are image-feature groups, **not identified tree species**. With 55 crowns in six spatially clustered circles, cluster stability and significance are weak; use them to generate hypotheses for botanical field verification, not to assert Nadine mixed two species.

In [ ]:
# NBAA Section 11 — label-free cluster discovery; human-vote enrichment afterward.
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
FEATURE_COLUMNS=['area_m2','compactness','color_R','color_G','color_B',
  'brightness_CV','RGB_texture','DSM_roughness_m','DSM_range_p90_p10_m']
d=FEATURES.merge(matched[['verify_id',*EXPERTS]],on='verify_id',validate='one_to_one')
available=[col for col in FEATURE_COLUMNS if d[col].notna().sum()>=max(12,int(.6*len(d)))]
assert len(available)>=5,('Insufficient crown features',available)
X=RobustScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(
    d[available].replace([np.inf,-np.inf],np.nan)))
coords=PCA(n_components=2).fit_transform(X)
d['PC1'],d['PC2']=coords[:,0],coords[:,1]
solutions=[]
for k in (2,3,4):
    model=KMeans(n_clusters=k,random_state=2026,n_init=25).fit(X)
    solutions.append((k,float(silhouette_score(X,model.labels_)),model.labels_))
print('Unsupervised k sensitivity (k,silhouette):',[(k,round(sc,3)) for k,sc,_ in solutions])
k,sc,labels=max(solutions,key=lambda x:x[1])
d['cluster']=labels
fig,axes=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for cluster,part in d.groupby('cluster'):
    axes[0].scatter(part.PC1,part.PC2,label=f'cluster {cluster}, n={len(part)}',alpha=.85)
for circle,part in d.groupby('circle_id'):
    axes[1].scatter(part.PC1,part.PC2,label=f'circle {circle}',alpha=.85)
for ax in axes:
    ax.set(xlabel='PCA 1',ylabel='PCA 2')
    ax.legend(fontsize=8)
axes[0].set_title('Crown clusters (no human labels used)')
axes[1].set_title('Identical PCA, coloured by circle')
fig.savefig(OUT/'NBAA_11_clusters_vs_circle.png',dpi=180)
plt.show();plt.close(fig)
profiles=[]
for cl,part in d.groupby('cluster'):
    for person in EXPERTS:
        profiles.append(dict(cluster=int(cl),annotator=person,n=int(len(part)),
           n_M=int((part[person]=='M').sum()),n_N=int((part[person]=='N').sum()),
           n_U=int((part[person]=='U').sum()),
           proportion_M=float((part[person]=='M').mean())))
PROFILES=pd.DataFrame(profiles)
PROFILES.to_csv(OUT/'NBAA_11_cluster_by_annotator.csv',index=False)
d.to_csv(OUT/'NBAA_11_cluster_crown_assignments.csv',index=False)
display(PROFILES.pivot(index='cluster',columns='annotator',values='proportion_M').round(3))
print('Cluster composition by circle (important confounding test):')
display(pd.crosstab(d.cluster,d.circle_id))
print('Chosen exploratory k=',k,'silhouette=',round(sc,3))
print('Next: inspect matched RGB crown images by cluster before any species hypothesis.')

## 12 — Within-annotator image consistency: spatially separated RGB U-Net training

**Primary question:** Can an image model reproduce each annotator's own Musanga crown delineations in *other image regions within the same annotated circles*? This is **not a test of transfer to another forest or observer accuracy**.

- One independent **ImageNet ResNet18 U-Net per annotator** (Nadine, Thomas, Marijn, Joseph), reusing the V3_1 architecture but never Nadine-trained weights.
- Use **every circle mapped by that annotator**. Circles with no polygons remain *unassessed*, never negative.
- For mapped circles, the annotation objective was exhaustive; within valid RGB ∩ original circle ∩ original ROI, pixels inside that annotator's Musanga polygons are positive, remaining pixels negative. This is an explicit **assumed-exhaustiveness** contract (draft metadata still does not independently confirm completion).
- Split image space **within each circle** into training / validation regions. Validation is a compact group of approximately 20% of non-overlapping 256-pixel tiles, anchored on at least one positive crown, with a **full-chip-width exclusion buffer**: no training chip can intersect the validation tiles and vice versa. This prevents overlapping-chip leakage. Validation patches are non-overlapping; training patches may overlap one another. Rare-positive circles that cannot be independently split are **training-only**, with no fabricated validation score.
- More training examples (up to 160 train chips and 64 validation chips per circle, cached as disk-backed NumPy arrays), moderately positive-enriched sampling, **BCE-only loss**, maximum 30 epochs, early stopping and **3 reproducible seeds**. Select checkpoints using held-out validation BCE. Fixed 0.5 threshold is the primary score; threshold sweeps are **diagnostic only**, since selecting thresholds on the evaluation data would be optimistic.
- Evaluate per circle (F1, IoU, precision, recall, cover bias), plus per-annotator **circle-macro** averages and seed-to-seed variability. These are validation estimates and are subject to checkpoint-selection optimism; no claims about inter-forest transfer.
- Keep Sections 00–11 and the original 55-crown classifications unchanged. Save results in a new memory-safe output subfolder and do not overwrite old LOSO checkpoints.

**Practical limitation:** With only a few positive crowns per circle, spatial validation can contain very few independent crowns; inspect positive counts and held-out visual examples before interpreting differences between people.

In [ ]:
# 12A — Source audit, per-annotator eligibility, reproducible CNN setup.
%pip -q install segmentation-models-pytorch
import os, hashlib, time, random, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from IPython.display import display
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from rasterio.features import rasterize
from rasterio.windows import Window
from shapely.geometry import mapping
import segmentation_models_pytorch as smp

# All modelling libraries are explicitly imported here; the upstream *data*
# objects are still provided by Sections 03, 07 and 08.
required_upstream=['ROOT','OUT','audit','SURVEY_CIRCLES','UTM','circles',
                   'POLY','OVERVIEW_QA','RGB_SOURCES']
missing=[v for v in required_upstream if v not in globals()]
if missing:
    raise RuntimeError(
        'NBAA modelling requires the data setup / geometry / RGB source cells. '
        'Run Sections 00, 03, 07 and 08 first; missing: '+', '.join(missing))
SEED=20261106
CHIP=256
TRAIN_STRIDE=128
MAX_TRAIN_CHIPS_PER_CIRCLE=160
MAX_VAL_CHIPS_PER_CIRCLE=64
EPOCHS=30
PATIENCE=6
REPEAT_SEEDS=[20261106,20261107,20261108]
VALIDATION_FRACTION=0.20
BATCH=4
LR=2e-4
THRESHOLD=.50
TRAIN_ANNOTATORS=['Nadine','Thomas','Marijn','Joseph']
GPU='cuda' if torch.cuda.is_available() else 'cpu'
assert GPU=='cuda','Select Colab Runtime > Change runtime type > GPU for U-Net experiment.'
torch.manual_seed(SEED);np.random.seed(SEED);random.seed(SEED)
torch.backends.cudnn.benchmark=True
UNET_OUT=OUT/'12_annotator_unet_within_circle_v3_memorysafe'
CHIP_OUT=UNET_OUT/'chips'
MODEL_OUT=UNET_OUT/'checkpoints'
for x in [UNET_OUT,CHIP_OUT,MODEL_OUT]:x.mkdir(parents=True,exist_ok=True)

# Read audited region-of-interest; do not redefine the V3_1 spatial support.
ROI_SOURCE=Path(audit['roi'])
assert ROI_SOURCE.is_file()
ROI_SHAPES=gpd.read_file(ROI_SOURCE)
assert ROI_SHAPES.crs is not None
CIRCLE_MAP={int(row.circle_id):row.geometry for _,row in SURVEY_CIRCLES.to_crs(UTM).iterrows()}
assert set(circles).issubset(CIRCLE_MAP.keys())
CIRCLE_MOSAIC={int(x.circle_id):x.source for x in OVERVIEW_QA.itertuples(index=False)}
assert set(circles).issubset(CIRCLE_MOSAIC)
CIRCLE_STATUS=pd.DataFrame([
    dict(annotator=p,circle_id=int(cid),n_positive_polygons=int(
        (POLY[p].circle_id==cid).sum()),
         status='assumed_exhaustive' if (POLY[p].circle_id==cid).any() else 'not_assessed')
    for p in TRAIN_ANNOTATORS for cid in circles])
CIRCLE_STATUS.to_csv(UNET_OUT/'12_circle_annotator_exhaustiveness_assumption.csv',index=False)
display(CIRCLE_STATUS.pivot(index='circle_id',columns='annotator',values='status'))
print('Mapped circles per annotator:',CIRCLE_STATUS[CIRCLE_STATUS.status=='assumed_exhaustive']
      .groupby('annotator').circle_id.count().to_dict())
print('GPU:',torch.cuda.get_device_name(0))
print('U-Net is trained independently for each annotator; no V3_1 weights are loaded.')


In [ ]:
# 12B — Cached 256 x 256 native RGB chips; supervised pixels strictly within circle ∩ ROI.
from rasterio.features import rasterize
from shapely.ops import unary_union
import hashlib

def area_gdf_in_crs(gdf,crs):
    return gdf.to_crs(crs).geometry

def pixel_window_clipped(ds,col,row):
    col=max(0,min(int(col),ds.width-CHIP))
    row=max(0,min(int(row),ds.height-CHIP))
    return Window(col,row,CHIP,CHIP)

def source_geometry(person,cid,crs):
    # No annotation from other experts is included in this polygon set.
    part=POLY[person].loc[POLY[person].circle_id==cid]
    return part.to_crs(crs).geometry.tolist()

def window_mask(geoms,window,transform):
    if not geoms:return np.zeros((int(window.height),int(window.width)),dtype=bool)
    return rasterize(((g,1) for g in geoms if g is not None and not g.is_empty),
             out_shape=(int(window.height),int(window.width)),
             transform=rasterio.windows.transform(window,transform),
             fill=0,dtype='uint8').astype(bool)

# 12B — Candidate-chip audit: a contiguous, non-overlapping spatial holdout
# per mapped circle. Keep spatial splitting identical for all seeds.
from shapely.geometry import box as geom_box

def validation_tile_plan(person,cid,ds,circle,positive_geoms,support):
    # Use global nonoverlapping 256 px cells, not an arbitrary thin stripe.
    # Select ~20% contiguous cells near one positive crown, and exclude
    # ALL overlapping training chips, so train and val see disjoint pixels.
    bounds=rasterio.windows.from_bounds(*circle.bounds,transform=ds.transform)
    x0=max(0,int(np.floor(bounds.col_off/CHIP)*CHIP))
    y0=max(0,int(np.floor(bounds.row_off/CHIP)*CHIP))
    x1=min(ds.width-CHIP,int(np.ceil((bounds.col_off+bounds.width)/CHIP)*CHIP))
    y1=min(ds.height-CHIP,int(np.ceil((bounds.row_off+bounds.height)/CHIP)*CHIP))
    candidates=[]
    for rr in range(y0,y1+1,CHIP):
        for cc in range(x0,x1+1,CHIP):
            w=Window(cc,rr,CHIP,CHIP)
            if window_mask([support],w,ds.transform).sum()>=400:
                hit=window_mask(positive_geoms,w,ds.transform)
                candidates.append((cc,rr,int(hit.sum())))
    if len(candidates)<3:
        raise ValueError((person,cid,'Only',len(candidates),
            'spatial tiles; 256px spatial validation unsupported'))
    # A positive-containing tile with the fewest crown pixels minimizes
    # removal of scarce positives from the training set.
    positive_tiles=[t for t in candidates if t[2]>=20]
    if not positive_tiles:raise ValueError((person,cid,'no positive validation tiles'))
    anchor=min(positive_tiles,key=lambda t:(t[2],t[1],t[0]))
    nval=max(1,int(round(len(candidates)*VALIDATION_FRACTION)))
    # A compact contiguous region, anchored on a Musanga-positive tile.
    ranked=sorted(candidates,key=lambda t:(
        (t[0]-anchor[0])**2+(t[1]-anchor[1])**2,t[1],t[0]))
    selected={(cc,rr) for cc,rr,_ in ranked[:nval]}
    return selected,len(candidates),anchor

def area_gdf_in_crs(gdf,crs):return gdf.to_crs(crs).geometry

def source_signature(person,cid):
    h=hashlib.sha256()
    for g in POLY[person].loc[POLY[person].circle_id==cid].geometry:h.update(g.wkb)
    h.update(CIRCLE_MAP[cid].wkb)
    p=RGB_SOURCES[CIRCLE_MOSAIC[cid]]
    s=p.stat()
    h.update(str((p.name,s.st_size,s.st_mtime_ns,CHIP,TRAIN_STRIDE,
                  MAX_TRAIN_CHIPS_PER_CIRCLE,MAX_VAL_CHIPS_PER_CIRCLE,
                  VALIDATION_FRACTION,'disklazy_v4')).encode())
    return h.hexdigest()[:20]

def create_circle_chips(person,cid):
    # Flat disk-backed .npy arrays; no in-memory aggregation of all chips.
    # Cache is valid only after the complete manifest has been written.
    cache=CHIP_OUT/f'{person}_circle_{cid:02d}_{source_signature(person,cid)}'
    manifest=cache/'manifest.json'
    if manifest.is_file():
        d=json.loads(manifest.read_text())
        if (d.get('contract')=='NBAA_spatial_disklazy_v4'
            and all((cache/f'{split}_{key}.npy').is_file()
                    for split in ('train','val') for key in ('rgb','y','w'))):
            return cache
    assert (POLY[person].circle_id==cid).any(),('Unassessed circle',person,cid)
    name=CIRCLE_MOSAIC[cid]
    with rasterio.open(RGB_SOURCES[name]) as ds:
        assert ds.count>=3 and ds.width>=CHIP and ds.height>=CHIP
        circle=gpd.GeoSeries([CIRCLE_MAP[cid]],crs=UTM).to_crs(ds.crs).iloc[0]
        roi=unary_union(area_gdf_in_crs(ROI_SHAPES,ds.crs).tolist())
        support=circle.intersection(roi)
        assert not support.is_empty,(person,cid,'circle outside original ROI')
        positives=source_geometry(person,cid,ds.crs)
        val_keys,total_tiles,anchor=validation_tile_plan(person,cid,ds,circle,positives,support)
        training_only=(len(positives)<2)  # Cannot independently split one positive crown
        val_footprints=[geom_box(*rasterio.windows.bounds(
            Window(cc,rr,CHIP,CHIP),ds.transform)) for cc,rr in val_keys]
        bounds=rasterio.windows.from_bounds(*circle.bounds,transform=ds.transform)
        col_lo=max(0,int(np.floor(bounds.col_off/CHIP)*CHIP))
        row_lo=max(0,int(np.floor(bounds.row_off/CHIP)*CHIP))
        col_hi=min(ds.width-CHIP,int(np.ceil((bounds.col_off+bounds.width)/CHIP)*CHIP))
        row_hi=min(ds.height-CHIP,int(np.ceil((bounds.row_off+bounds.height)/CHIP)*CHIP))
        def intersects_positive(w):
            bb=geom_box(*rasterio.windows.bounds(w,ds.transform))
            return any(g.intersects(bb) for g in positives)
        def train_allowed(w):
            bb=geom_box(*rasterio.windows.bounds(w,ds.transform))
            return not any(bb.intersects(g) for g in val_footprints)
        # Window coordinates only; no RGB or binary masks are read yet.
        positive_windows=[];background_windows=[]
        for rr in range(row_lo,row_hi+1,TRAIN_STRIDE):
            for cc in range(col_lo,col_hi+1,TRAIN_STRIDE):
                w=Window(cc,rr,CHIP,CHIP)
                if not train_allowed(w):continue
                if intersects_positive(w):positive_windows.append((cc,rr))
                else:background_windows.append((cc,rr))
        # Ensure crown-centred windows are candidates even if off the grid.
        for g in positives:
            rr,cc=ds.index(g.centroid.x,g.centroid.y)
            c=max(0,min(ds.width-CHIP,cc-CHIP//2))
            r=max(0,min(ds.height-CHIP,rr-CHIP//2))
            if train_allowed(Window(c,r,CHIP,CHIP)):
                positive_windows.append((c,r))
        positive_windows=list(dict.fromkeys(positive_windows))
        background_windows=list(dict.fromkeys(background_windows))
        rng=np.random.default_rng(SEED+cid)
        rng.shuffle(positive_windows);rng.shuffle(background_windows)
        # Moderate positive enrichment, but do not read rejected candidates.
        npos=min(len(positive_windows),MAX_TRAIN_CHIPS_PER_CIRCLE//2)
        train_windows=positive_windows[:npos]+background_windows[
            :MAX_TRAIN_CHIPS_PER_CIRCLE-npos]
        if len(train_windows)<MAX_TRAIN_CHIPS_PER_CIRCLE:
            train_windows+=positive_windows[npos:npos+MAX_TRAIN_CHIPS_PER_CIRCLE-len(train_windows)]
        val_windows=sorted(val_keys)
        if training_only:
            train_windows=list(dict.fromkeys(train_windows+val_windows))
            val_windows=[]
        if len(val_windows)>MAX_VAL_CHIPS_PER_CIRCLE:
            # Uniform subset for prevalence; require at least one positive tile.
            positive_val=[v for v in val_windows if intersects_positive(Window(*v,CHIP,CHIP))]
            selected=rng.choice(len(val_windows),size=MAX_VAL_CHIPS_PER_CIRCLE,replace=False)
            val_windows=[val_windows[int(i)] for i in sorted(selected)]
            if positive_val and not any(v in positive_val for v in val_windows):
                val_windows[-1]=positive_val[0]
        def write_split(split,windows):
            # Allocate on disk; the returned .npy files are memory-mapped on demand.
            cache.mkdir(parents=True,exist_ok=True)
            files={k:np.lib.format.open_memmap(cache/f'{split}_{k}.npy',mode='w+',
                        dtype='uint8',shape=(max(1,len(windows)),CHIP,CHIP,3) if k=='rgb'
                        else (max(1,len(windows)),CHIP,CHIP)) for k in ('rgb','y','w')}
            valid_count=0;positive_count=0;positive_pixels=0
            for cc,rr in windows:
                w=Window(cc,rr,CHIP,CHIP)
                covered=window_mask([support],w,ds.transform)
                if covered.sum()<400:continue
                raw=ds.read([1,2,3],window=w,masked=True)
                img=raw.filled(0)
                valid=(~np.ma.getmaskarray(raw)).all(axis=0)
                valid &= np.any(img!=0,axis=0)&~((img[0]==0)&(img[1]==0)&(img[2]==255))
                weight=(valid&covered).astype('uint8')
                if weight.sum()<400:continue
                y=(window_mask(positives,w,ds.transform)&covered).astype('uint8')
                files['rgb'][valid_count]=np.clip(img,0,255).astype('uint8').transpose(1,2,0)
                files['y'][valid_count]=y
                files['w'][valid_count]=weight
                positive_pixels+=int((y*weight).sum())
                positive_count+=int((y*weight).sum()>0)
                valid_count+=1
            for mem in files.values():mem.flush()
            del files
            if valid_count==0 and split=='train':raise ValueError((person,cid,split,'no valid RGB chips'))
            return dict(n=valid_count,positive_chips=positive_count,positive_pixels=positive_pixels)
        stats_train=write_split('train',train_windows)
        stats_val=write_split('val',val_windows)
        if not stats_train['positive_pixels'] or (val_windows and not stats_val['positive_pixels']):
            # With rare positives a spatial split may leave one side empty.
            # Use the whole mapped circle for training, never invent a validation score.
            print(f'{person} C{cid}: impossible positive-aware split; training-only.',flush=True)
            training_only=True
            train_windows=list(dict.fromkeys(train_windows+val_windows))
            val_windows=[]
            stats_train=write_split('train',train_windows)
            stats_val=write_split('val',[])
        assert stats_train['positive_pixels']>0,(person,cid,'no training positive pixels')
        info=dict(contract='NBAA_spatial_disklazy_v4',person=person,circle_id=int(cid),
                  split_tiles=len(val_keys),total_tiles=total_tiles,
                  train=stats_train,val=stats_val,
                  validation_eligible=bool(stats_val['positive_pixels'] and not training_only),
                  split_status='training_only' if training_only else 'spatial_validation')
        manifest.write_text(json.dumps(info,indent=2))
        print(
            f'{person} C{cid} [{info["split_status"]}]: '
            f'train={stats_train["n"]}, val={stats_val["n"]}, '
            f'positive tiles train/val={stats_train["positive_chips"]}/'
            f'{stats_val["positive_chips"]}',
            flush=True
        )
        return cache

class SegChips(Dataset):
    def __init__(self,paths,split,augment=False):
        # Store only chip indices. Open .npy arrays as read-only memmaps on demand.
        self.items=[];self.augment=augment
        self.mmaps=[]
        for folder in paths:
            metadata=json.loads((folder/'manifest.json').read_text())
            count=int(metadata[split]['n'])
            mats={k:np.load(folder/f'{split}_{k}.npy',mmap_mode='r')
                  for k in ('rgb','y','w')}
            assert all(m.shape[0]>=count for m in mats.values())
            group=len(self.mmaps);self.mmaps.append(mats)
            self.items.extend((group,j) for j in range(count))
    def __len__(self):return len(self.items)
    def __getitem__(self, i):
        group, idx = self.items[i]
        arrays = self.mmaps[group]
        rgb = np.array(arrays["rgb"][idx], copy=True)
        target = np.array(arrays["y"][idx], copy=True)
        weight = np.array(arrays["w"][idx], copy=True)

        if self.augment:
            k = int(torch.randint(0, 4, (1,)).item())
            rgb = np.rot90(rgb, k)
            target = np.rot90(target, k)
            weight = np.rot90(weight, k)
            if torch.rand(()).item() < 0.5:
                rgb = rgb[:, ::-1]
                target = target[:, ::-1]
                weight = weight[:, ::-1]
            if torch.rand(()).item() < 0.5:
                rgb = rgb[::-1]
                target = target[::-1]
                weight = weight[::-1]

        image_tensor = torch.from_numpy(
            np.array(rgb.transpose(2, 0, 1), copy=True, order="C")
        ).float() / 255.0
        target_tensor = torch.from_numpy(
            np.array(target, copy=True, order="C")
        ).float()
        weight_tensor = torch.from_numpy(
            np.array(weight, copy=True, order="C")
        ).float()
        return image_tensor, target_tensor, weight_tensor

class AnnotatorUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=smp.Unet(encoder_name='resnet18',encoder_weights='imagenet',in_channels=3,classes=1)
    def forward(self,x):return self.net(x)[:,0]

def masked_bce(logits,y,w):
    loss=F.binary_cross_entropy_with_logits(logits.float(),y,reduction='none')
    return (loss*w).sum()/w.sum().clamp_min(1.)
print('Spatial within-circle chip preparation ready (new cache contract).')

In [ ]:
# 12C — One U-Net per annotator and seed, all mapped circles in TRAIN/VAL split.
# Checkpoint selection is based on validation BCE; resumed runs reuse checkpoints.
from collections import defaultdict
RUN_ANNOTATORS=['Nadine','Thomas','Marijn','Joseph']
RUN_SEEDS=REPEAT_SEEDS[:]            # set RUN_SEEDS=REPEAT_SEEDS[:1] for smoke test
RUN_TRAINING=True
USE_AMP=(GPU=='cuda')
TRAIN_REGISTRY=UNET_OUT/'12_within_circle_seed_registry.csv'

def train_person_seed(person,seed):
    eligible=sorted(CIRCLE_STATUS.query('annotator == @person and status == "assumed_exhaustive"').circle_id.astype(int))
    assert len(eligible)>=3,(person,eligible)
    keys=[(cid,source_signature(person,cid)) for cid in eligible]
    fingerprint=hashlib.sha256(str((person,seed,keys,EPOCHS,PATIENCE,LR,BATCH,'BCE_v2')).encode()).hexdigest()[:16]
    ckpt=MODEL_OUT/f'{person}_seed_{seed}_{fingerprint}.pt'
    hist=MODEL_OUT/f'{person}_seed_{seed}_{fingerprint}_history.csv'
    if ckpt.exists():
        print('Reusing',ckpt.name,flush=True)
        return dict(annotator=person,seed=seed,checkpoint=str(ckpt),
                    circles=','.join(map(str,eligible)),history=str(hist))
    paths=[create_circle_chips(person,cid) for cid in eligible]
    train=SegChips(paths,'train',augment=True)
    val_paths=[p for p in paths if json.loads((p/'manifest.json').read_text())
               .get('validation_eligible',True)]
    val=SegChips(val_paths,'val',augment=False)
    assert len(train)>0 and len(val)>0
    tr_loader=DataLoader(train,batch_size=BATCH,shuffle=True,pin_memory=True,num_workers=0)
    va_loader=DataLoader(val,batch_size=BATCH,shuffle=False,pin_memory=True,num_workers=0)
    torch.manual_seed(seed);np.random.seed(seed);random.seed(seed)
    model=AnnotatorUNet().to(GPU)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)
    best=float('inf');wait=0;history=[]
    for epoch in range(EPOCHS):
        model.train();losses=[]
        for xx,yy,ww in tr_loader:
            xx=xx.to(GPU,non_blocking=True);yy=yy.to(GPU,non_blocking=True);ww=ww.to(GPU,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16,enabled=USE_AMP):
                logits=model(xx)
            loss=masked_bce(logits,yy,ww)
            scaler.scale(loss).backward();scaler.step(opt);scaler.update()
            losses.append(float(loss.detach()))
        model.eval();vsum=0.;vweight=0.
        with torch.inference_mode():
            for xx,yy,ww in va_loader:
                xx=xx.to(GPU);yy=yy.to(GPU);ww=ww.to(GPU)
                with torch.autocast('cuda',dtype=torch.float16,enabled=USE_AMP):
                    logits=model(xx)
                cur=F.binary_cross_entropy_with_logits(logits.float(),yy,reduction='none')
                vsum+=float((cur*ww).sum().cpu());vweight+=float(ww.sum().cpu())
        validation_bce=vsum/max(vweight,1)
        history.append(dict(annotator=person,seed=seed,epoch=epoch+1,
                            train_BCE=float(np.mean(losses)),validation_BCE=validation_bce,
                            train_chips=len(train),val_chips=len(val)))
        pd.DataFrame(history).to_csv(hist,index=False)
        print(f'{person} seed {seed} epoch {epoch+1:02d} '
              f'train={history[-1]["train_BCE"]:.4f} val={validation_bce:.4f}',flush=True)
        if validation_bce<best-1e-4:
            best=validation_bce;wait=0
            torch.save(model.state_dict(),ckpt)
        else:
            wait+=1
            if wait>=PATIENCE:
                print('Early stopping',person,seed,'at',epoch+1,flush=True)
                break
    del model,train,val,tr_loader,va_loader
    torch.cuda.empty_cache()
    return dict(annotator=person,seed=seed,checkpoint=str(ckpt),
                circles=','.join(map(str,eligible)),history=str(hist))

registry=[]
if RUN_TRAINING:
    for person in RUN_ANNOTATORS:
        for seed in RUN_SEEDS:
            result=train_person_seed(person,seed)
            registry.append(result)
            pd.DataFrame(registry).to_csv(TRAIN_REGISTRY,index=False)
    print('Trained/resumed',len(registry),'annotator × seed models')
else:
    print('Training disabled; set RUN_TRAINING=True to fit models.')


## 13 — Within-circle held-out segmentation validation and seed stability

For each annotator-specific seed model, evaluate only its spatially disjoint **validation chips** from all of that annotator's mapped circles. Pixel-level precision, recall, F1, IoU, Brier score and predicted/observed cover are computed with a fixed 0.5 decision threshold. The same small validation set was used for checkpoint selection (early stopping), so these are **exploratory validation statistics**, not unbiased test estimates. Any optimized threshold is reported *only* as an explicitly optimistic sensitivity analysis.

Circle-macro summaries avoid allowing large circles to dominate; across-seed differences quantify model initialization sensitivity. Different annotators have different mapped-circle sets and differing positive prevalence, so performance is not an intrinsic ranking of their abilities.

In [ ]:
# 13A — Validation chips with full per-circle mask accounting, 3-seed stability.
from sklearn.metrics import roc_auc_score,average_precision_score
REGISTRY=pd.read_csv(TRAIN_REGISTRY) if TRAIN_REGISTRY.exists() else pd.DataFrame()
assert len(REGISTRY),'Train at least one seed in Section 12C before evaluation.'
EVAL_FILE=UNET_OUT/'13_within_circle_per_seed_per_circle.csv'
prior=pd.read_csv(EVAL_FILE) if EVAL_FILE.exists() else pd.DataFrame()
results=prior.to_dict('records')

def evaluate_val_chips(model,person,cid,seed):
    path=create_circle_chips(person,cid)
    metadata=json.loads((path/'manifest.json').read_text())
    if not metadata.get('validation_eligible',True):return None
    val=SegChips([path],'val',augment=False)
    loader=DataLoader(val,batch_size=BATCH,shuffle=False,num_workers=0,pin_memory=True)
    tp=fp=fn=tn=0;n=0;observed=0;predicted=0;sq=0
    ys=[];ps=[];rng=np.random.default_rng(SEED+cid+seed)
    auc_cap=30000
    model.eval()
    with torch.inference_mode():
        for xx,yy,ww in loader:
            xx=xx.to(GPU)
            with torch.autocast('cuda',dtype=torch.float16,enabled=USE_AMP):
                p=torch.sigmoid(model(xx)).float().cpu().numpy()
            y=yy.numpy();w=ww.numpy()>0
            z=p[w];truth=y[w]>0.5;hat=z>=.5
            tp+=int(np.sum(hat&truth));fp+=int(np.sum(hat&~truth))
            fn+=int(np.sum(~hat&truth));tn+=int(np.sum(~hat&~truth))
            n+=len(z);observed+=int(truth.sum());predicted+=float(z.sum())
            sq+=float(np.square(z-truth).sum())
            take=rng.choice(len(z),size=min(len(z),max(100,auc_cap//max(1,len(loader)))),replace=False)
            ys.extend(truth[take].astype('uint8').tolist());ps.extend(z[take].tolist())
    assert n>0
    ys=np.asarray(ys,dtype='uint8');ps=np.asarray(ps,dtype='float32')
    return dict(annotator=person,circle_id=cid,seed=int(seed),
        val_chips=len(val),val_pixels=n,observed_cover=observed/n,
        predicted_cover=predicted/n,cover_bias_pp=100*(predicted-observed)/n,
        f1=2*tp/max(1,2*tp+fp+fn),iou=tp/max(1,tp+fp+fn),
        precision=tp/(tp+fp) if tp+fp else np.nan,
        recall=tp/(tp+fn) if tp+fn else np.nan,
        brier=sq/n,tp=tp,fp=fp,fn=fn,tn=tn,
        auc=roc_auc_score(ys,ps) if len(np.unique(ys))>1 else np.nan,
        ap=average_precision_score(ys,ps) if len(np.unique(ys))>1 else np.nan)

for r in REGISTRY.itertuples(index=False):
    ckpt=Path(r.checkpoint)
    assert ckpt.is_file(),ckpt
    model=AnnotatorUNet().to(GPU)
    model.load_state_dict(torch.load(ckpt,map_location=GPU,weights_only=True))
    for cid in [int(x) for x in str(r.circles).split(',')]:
        exists=(len(prior)>0 and ((prior.annotator==r.annotator)&
                (prior.circle_id==cid)&(prior.seed==int(r.seed))&
                (prior.checkpoint==str(ckpt))).any())
        if exists:
            print('Cached:',r.annotator,r.seed,cid);continue
        q=evaluate_val_chips(model,r.annotator,cid,int(r.seed))
        if q is None:
            print('Training-only circle excluded from validation:',r.annotator,cid)
            continue
        q['checkpoint']=str(ckpt)
        results=[x for x in results if not (x['annotator']==r.annotator and
                 int(x['circle_id'])==cid and int(x['seed'])==int(r.seed))]
        results.append(q)
        pd.DataFrame(results).to_csv(EVAL_FILE,index=False)
        print(r.annotator,'seed',r.seed,'circle',cid,'F1',round(q['f1'],3),
              'IoU',round(q['iou'],3),'cover bias',round(q['cover_bias_pp'],2),flush=True)
    del model;torch.cuda.empty_cache()

EVAL=pd.DataFrame(results)
display(EVAL[['annotator','circle_id','seed','val_chips','observed_cover',
              'f1','iou','precision','recall','cover_bias_pp','auc']].round(3))
seed_macro=EVAL.groupby(['annotator','seed'],as_index=False).agg(
    circles=('circle_id','nunique'),mean_F1=('f1','mean'),
    mean_IoU=('iou','mean'),mean_precision=('precision','mean'),
    mean_recall=('recall','mean'),mean_cover_bias_pp=('cover_bias_pp','mean'))
display(seed_macro.round(3))
stability=seed_macro.groupby('annotator').agg(
    seeds=('seed','nunique'),
    mean_F1=('mean_F1','mean'),sd_F1=('mean_F1','std'),
    mean_IoU=('mean_IoU','mean'),sd_IoU=('mean_IoU','std'),
    mean_cover_bias_pp=('mean_cover_bias_pp','mean'),
    sd_cover_bias_pp=('mean_cover_bias_pp','std'))
display(stability.round(3))
seed_macro.to_csv(UNET_OUT/'13_macro_by_seed.csv',index=False)
stability.to_csv(UNET_OUT/'13_seed_stability.csv')
print('Validation data selected the checkpoint: these scores are exploratory, not unbiased test performance.')


## 14 — How to interpret the within-circle experiment

1. Main result: mean and variability of **spatially held-out, within-circle** crown-mask F1/IoU across seeds. No transfer to a new forest is required.
2. The validation region is a contiguous stripe with training windows excluded wherever their footprint touches the held-out stripe. Check that each split has positive pixels; only spatially supported chips are scored.
3. Inspect predictions over the *same RGB imagery* and compare positive crown types to identify annotator-specific visual selection. A model may be stable yet reproduce an incomplete or biased annotation concept.
4. There are only four–six independently mapped circles per annotator, and the validation set also guides early stopping. Treat all results as **feasibility / internal predictability**, not proof of botanical correctness or formal annotator repeatability.
5. The five-expert reference classifications remain separate (Sections 01–02 and 09); uncertain votes are never converted to negatives.
6. Original LOSO results and checkpoints remain in the older `12_annotator_unet` output folder for optional transfer comparison. New outputs use `12_annotator_unet_within_circle_v2`.

Next scientific analysis: overlay each annotator's within-circle model probabilities onto identical native-resolution RGB crown crops, especially crowns accepted only by Nadine.